In [1]:
import random
import math

In [2]:
def safe_div(x, y): return x / y if abs(y) > 1e-6 else 1
def safe_sqrt(x): return math.sqrt(abs(x))
def safe_log(x): return math.log(abs(x)) if abs(x) > 1e-6 else 0
def safe_pow(x, y):
    try:
        res = x ** y
        if isinstance(res, complex): return res.real
        return res
    except: return 1

FUNCTIONS = {
    "add": (lambda x, y: x + y, 2),
    "sub": (lambda x, y: x - y, 2),
    "mul": (lambda x, y: x * y, 2),
    "div": (safe_div, 2),
    "pow": (safe_pow, 2),
    "sqrt": (safe_sqrt, 1),
    "sin": (lambda x: math.sin(x), 1),
    "cos": (lambda x: math.cos(x), 1),
    "tan": (lambda x: math.tan(x), 1),
    "abs": (lambda x: abs(x), 1),
    "log": (safe_log, 1),
}

In [3]:
class Node:
    def __init__(self, value, children = None):
        self.value = value
        self.children = children if children else []

    def evaluate(self, input_value):
        if self.value == 'x':
            return input_value
        if isinstance(self.value, (int, float)):
            return self.value

        func, _ = FUNCTIONS[self.value]
        args = [child.evaluate(input_value) for child in self.children]

        try:
            return func(*args)
        except:
            return 0

    def __str__(self):
        if not self.children:
            return str(self.value)
        else:
            return f"({str(self.value)} {' '.join(map(str, self.children))})"

In [4]:
def init_random_tree(depth: int):
    if depth == 0 or random.random() < 0.3:
        value = 'x' if random.random() < 0.7 else round(random.uniform(-2, 2), 1)
        return Node(value=value)

    func_key = random.choice(list(FUNCTIONS.keys()))
    _, children_count = FUNCTIONS[func_key]

    children = [init_random_tree(depth - 1) for _ in range(children_count)]

    return Node(func_key, children)

In [5]:
def copy_tree(node: Node):
    return Node(node.value, [copy_tree(c) for c in node.children])

def get_depth(node: Node):
    if not node.children:
        return 1
    return 1 + max([get_depth(child) for child in node.children])

def get_node_level(root: Node, target_node: Node, current_level: int = 1):
    if target_node == root:
        return current_level
    for child in root.children:
        level = get_node_level(child, target_node, current_level + 1)
        if level is not None:
            return level
    return None

def get_all_nodes(node):
    nodes = [node]
    for child in node.children:
        nodes.extend(get_all_nodes(child))
    return nodes

In [6]:
def cross_over(p1, p2, max_depth=7):
    c1 = copy_tree(p1)
    c2 = copy_tree(p2)

    n1, n2 = get_all_nodes(c1), get_all_nodes(c2)
    target1, target2 = random.choice(n1), random.choice(n2)

    # Hoán đổi
    target1.value, target2.value = target2.value, target1.value
    target1.children, target2.children = target2.children, target1.children

    # Kiểm tra độ sâu của CẢ CÂY mới
    res1 = c1 if get_depth(c1) <= max_depth else copy_tree(p1)
    res2 = c2 if get_depth(c2) <= max_depth else copy_tree(p2)

    return res1, res2

def mutate(node, mutation_rate=0.1, max_depth=7):
    child = copy_tree(node)

    if random.random() < mutation_rate:
        nodes_all = get_all_nodes(child)
        node_to_mutate = random.choice(nodes_all)
        current_level = get_node_level(child, node_to_mutate, 1)
        remaining_depth = max_depth - current_level

        if remaining_depth > 0:
            new_sub_tree = init_random_tree(depth=random.randint(1, remaining_depth))
            node_to_mutate.value = new_sub_tree.value
            node_to_mutate.children = new_sub_tree.children

    return child

In [7]:
def create_next_children_generation(
        parents,
        cross_over_func,
        mutation_func,
        mutation_rate=0.01,
        max_depth=7
):
    n = len(parents)
    parents = random.sample(parents, n)
    children = []

    for i in range(0, n-1, 2):
        p1 = parents[i]
        p2 = parents[i+1]
        c1, c2 = cross_over_func(p1, p2, max_depth)
        c1 = mutation_func(c1, mutation_rate, max_depth)
        c2 = mutation_func(c2, mutation_rate, max_depth)
        children.append(c1)
        children.append(c2)

    return children

In [8]:
def calculate_fitness(nodes, dataset):
    mse_ans = []
    for node in nodes:
        mse = 0
        for x, y_true in dataset:
            try:
                y_pred = node.evaluate(x)
                mse += (y_true - y_pred) ** 2
            except:
                mse += 1e3
        penalty = len(get_all_nodes(node)) * 0.1
        mse_ans.append(mse / len(dataset) + penalty)

    return mse_ans

In [9]:
def select_node(population, fitnesses, k=3):
    init_idx = random.sample(range(len(population)), k)
    best_idx = min(init_idx, key=lambda i: fitnesses[i])
    return population[best_idx]

def select_nodes_for_next_generation(
        population,
        fitnesses,
        population_size=10,
        k=3
):
    next_generation = [select_node(population, fitnesses, k) for _ in range(population_size)]
    return next_generation

In [10]:
def best_selection_gp(
        population,
        fitnesses
):
    best_idx = min(range(len(population)), key=lambda i: fitnesses[i])
    return population[best_idx], fitnesses[best_idx]

In [11]:
def genetic_programming (
        target_func,
        dataset_points=100,
        population_size=100,
        max_generations=100,
        generation_func=create_next_children_generation,
        fitness_func=calculate_fitness,
        crossover_func=cross_over,
        selection_func=select_nodes_for_next_generation,
        mutation_func=mutate,
        best_selection_func=best_selection_gp,
        dataset_range=(-10, 10),
        stop_loss_value=0.01,
        mutation_rate=0.01,
        max_depth=9,
        init_depth=3,
        k=3
):
    step = (dataset_range[1] - dataset_range[0]) / dataset_points
    dataset = [(dataset_range[0] + i * step, target_func(dataset_range[0] + i * step)) for i in range(dataset_points)]
    nodes = [init_random_tree(init_depth) for _ in range(population_size)]
    current_gen = 0
    best_node = best_cost = None

    while True:
        print(f"Generation {current_gen}")

        fitnesses = fitness_func(nodes, dataset)

        best_node_local, nest_cost_local = best_selection_func(nodes, fitnesses)
        depth_local = get_depth(best_node_local)
        print(f"Best node local: {best_node_local} \n Cost local: {nest_cost_local} \n Depth local: {depth_local}")
        if best_cost is None or best_cost > nest_cost_local:
            best_node = best_node_local
            best_cost = nest_cost_local
            best_depth = depth_local
            print(f"New best node: {best_node}, cost: {best_cost} with depth: {best_depth}")

        if current_gen >= max_generations or best_cost <= stop_loss_value:
            break

        children = generation_func(nodes, crossover_func, mutation_func, mutation_rate, max_depth)
        full_next_generation = nodes + children
        full_fitnesses = fitness_func(full_next_generation, dataset)

        selection_nodes = selection_func(full_next_generation, full_fitnesses, population_size, k)
        nodes = selection_nodes

        current_gen += 1

    return best_node, best_cost

In [ ]:
def target_function(x):
    return x ** 2 + x + math.log(x ** 2 + 1) + math.sqrt(abs(x) ** 3) + 1

best_node, best_cost = genetic_programming(
    target_func=target_function,
    dataset_points=210,
    population_size=100,
    max_generations=1000,
    generation_func=create_next_children_generation,
    fitness_func=calculate_fitness,
    crossover_func=cross_over,
    selection_func=select_nodes_for_next_generation,
    mutation_func=mutate,
    best_selection_func=best_selection_gp,
    dataset_range=(-250, 250),
    stop_loss_value=0.1,
    mutation_rate=0.3,
    init_depth=3,
    k=3
)

print(">" * 50)
print(f"Best node: {best_node}")
print(f"Best cost: {best_cost}")

Generation 0
Best node local: (div (add (abs x) (add 1.7 x)) (log -1.6)), cost local: 877905174.569146 with depth: 4
New best node: (div (add (abs x) (add 1.7 x)) (log -1.6)), cost: 877905174.569146 with depth: 4
Generation 1
Best node local: (mul 1.7 (abs (sub x 0.2))), cost local: 881275272.6914871 with depth: 4
Generation 2
Best node local: (mul x x), cost local: 3953658.220673966 with depth: 2
New best node: (mul x x), cost: 3953658.220673966 with depth: 2
Generation 3
Best node local: (mul x x), cost local: 3953658.220673966 with depth: 2
Generation 4
Best node local: (mul x x), cost local: 3953658.220673966 with depth: 2
Generation 5
Best node local: (mul x x), cost local: 3953658.220673966 with depth: 2
Generation 6
Best node local: (add (abs (div x (abs (sub -0.3 x)))) (mul x x)), cost local: 3950480.4609382553 with depth: 6
New best node: (add (abs (div x (abs (sub -0.3 x)))) (mul x x)), cost: 3950480.4609382553 with depth: 6
Generation 7
Best node local: (add x (add (mul x x)